# SFU-EVP anomaly audit

Read-only audit of the current anomaly rules and candidate checks against the local dataset. The observed snapshot contains 91,518 sessions from 2019-11-06 to 2026-04-02 UTC. No authenticated API calls or dataset changes were made.

Run from the repository or its analysis directory with the runner dependencies installed. Counts are overlapping candidate flags, not independent fault incidents. Synthetic examples are explicitly labeled. Input SHA-256 hashes are included for reproducibility.


In [1]:
from pathlib import Path
import sys,json,hashlib
import numpy as np
import pandas as pd
ROOT=next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'lib/anomalies.py').exists() and (p/'data/Sessions.csv').exists())
sys.path.insert(0,str(ROOT))
from lib.anomalies import find_anomalies as _find_anomalies
def find_anomalies(sessions):
    return _find_anomalies(sessions, rules_version=1)
from lib.ChargePointDatasetUtils import read_csv
s=read_csv(ROOT/'data/Sessions.csv'); a=read_csv(ROOT/'data/Alarms.csv'); st=read_csv(ROOT/'data/Stations.csv'); old=read_csv(ROOT/'data/Anomalies.csv')
start=pd.to_numeric(s.start_ts,errors='coerce'); end=pd.to_numeric(s.end_ts,errors='coerce'); energy=pd.to_numeric(s.energy,errors='coerce')
duration=pd.to_timedelta(s.total_session_duration,errors='coerce').dt.total_seconds(); charging=pd.to_timedelta(s.total_charging_duration,errors='coerce').dt.total_seconds(); elapsed=end-start
calc=find_anomalies(s)
print('PROFILE',json.dumps({'rows':len(s),'period_start':str(pd.to_datetime(start.min(),unit='s',utc=True)),'period_end':str(pd.to_datetime(end.max(),unit='s',utc=True)),'stations':s.station_id.nunique(),'ports':len(st),'alarms':len(a),'current_rules':calc.anomaly_description.value_counts().to_dict(),'unique_flagged_sessions':calc.session_id.nunique()},default=str))
# Compare records independently of ordering and numeric CSV type inference.
x=calc.copy(); y=old.copy()
for d in [x,y]:
 d['value']=d['value'].astype(str)
for description in calc.anomaly_description.unique():
 xx=x[x.anomaly_description==description].sort_values('session_id').reset_index(drop=True)
 yy=y[y.anomaly_description==description].sort_values('session_id').reset_index(drop=True)
 if description=='Charging power exceeds 7 kW':
  equal=xx.session_id.equals(yy.session_id) and np.allclose(pd.to_numeric(xx.value),pd.to_numeric(yy.value))
 else: equal=xx.astype(str).equals(yy.astype(str))
 print('MATCH',description,equal)
checks={
 'duplicate_session_rows':s.session_id.duplicated(keep=False),
 'missing_session_id':s.session_id.eq(''),
 'invalid_start':start.isna(), 'invalid_end':end.isna(), 'invalid_energy':energy.isna(),
 'invalid_session_duration':duration.isna(),'invalid_charging_duration':charging.isna(),
 'negative_energy':energy<0,'end_before_start':elapsed<0,'zero_elapsed':elapsed==0,
 'negative_session_duration':duration<0,'negative_charge_duration':charging<0,
 'charge_exceeds_session_60sec':charging>duration+60,
 'session_duration_diff_elapsed_60sec':(duration-elapsed).abs()>60,
 'session_duration_diff_elapsed_5min':(duration-elapsed).abs()>300,
 'energy_positive_zero_charge_duration':(energy>0)&(charging==0),
 'energy_positive_zero_elapsed':(energy>0)&(elapsed==0),
 'zero_energy':energy==0,
 'zero_energy_session_5min':(energy==0)&(duration>=300),
 'zero_energy_session_30min':(energy==0)&(duration>=1800),
 'idle_4hours':(duration-charging)>=4*3600,
 'idle_12hours':(duration-charging)>=12*3600,
 'short_session_under_2min':(duration>=0)&(duration<120),
 'active_avg_power_gt7':(energy/(charging/3600)>7)&(charging>=36),
 'active_avg_power_gt7_5min':(energy/(charging/3600)>7)&(charging>=300),
 'future_end':end>pd.Timestamp.now(tz='UTC').timestamp(),
}
# Existing station power may describe current rather than historical capacity.
def ports(x): return x.astype(str).str.replace(r'\.0$','',regex=True)
lookup=st.assign(port_key=ports(st.port_no))
print('STATION_CAPACITY',lookup[['model','voltage','current','power']].drop_duplicates().to_dict('records'))
print('STATION_DUP_KEYS',int(lookup.duplicated(['station_id','port_key']).sum()))
joined=s.assign(port_key=ports(s.port_no)).merge(lookup[['station_id','port_key','power']],on=['station_id','port_key'],how='left',validate='many_to_one')
rating=pd.to_numeric(joined.power,errors='coerce')
checks['unknown_station_port']=rating.isna()
checks['active_avg_gt_rating_10pct_5min']=(energy/(charging/3600)>rating*1.1)&(charging>=300)
checks['elapsed_avg_gt_rating_10pct_5min']=(energy/(elapsed/3600)>rating*1.1)&(elapsed>=300)
# Half-open intervals, previous maximum end detects nested overlapping sessions.
ordered=s.assign(_start=start,_end=end,_port=ports(s.port_no)).sort_values(['station_id','_port','_start'])
previous_end=ordered.groupby(['station_id','_port'])['_end'].transform(lambda x:x.cummax().shift())
overlaps=(ordered._start<previous_end)&(ordered._end>ordered._start)
checks['overlapping_port_session']=overlaps.reindex(s.index,fill_value=False)
# Quick reattempts by the same pseudonymous driver at the same port.
r=s.assign(_start=start,_end=end,_port=ports(s.port_no)).sort_values('_start')
g=r.groupby(['user_id','station_id','_port']); gap=r._start-g._end.shift()
checks['same_driver_port_restart_within_5min']=((gap>=0)&(gap<=300)).reindex(s.index,fill_value=False)
print('CHECKS',json.dumps({k:{'count':int(v.sum()),'percent':round(100*v.mean(),3)} for k,v in checks.items()}))
print('YEARLY',pd.DataFrame({'year':pd.to_datetime(start,unit='s',utc=True).dt.year,'zero':checks['zero_energy_session_5min'],'count':1}).groupby('year').sum().to_dict('index'))
print('DURATION_DIFF', (duration-elapsed).describe(percentiles=[.5,.95,.99]).to_dict())
print('ALARM_TYPES',a.alarm_type.value_counts().head(12).to_dict())
print('ALARM_LINK_COVERAGE',int(a.session_id.ne('').sum()),len(a))
# Synthetic cases make silent failures explicit without altering source data.
base=dict(session_id='synthetic',start_ts=0,end_ts=3600,energy=1,total_session_duration='01:00:00',total_charging_duration='01:00:00')
for label, changes in [('negative_energy',{'energy':-1}),('reversed_time',{'end_ts':-1}),('malformed_duration',{'total_session_duration':'bad'}),('positive_energy_zero_time',{'end_ts':0,'total_session_duration':'00:00:00','total_charging_duration':'00:00:00'}),('high_active_power_hidden_by_idle',{'end_ts':36000,'energy':10,'total_session_duration':'10:00:00','total_charging_duration':'01:00:00'}),('exactly_24h',{'end_ts':86400,'total_session_duration':'24:00:00'})]:
 print('SYNTHETIC',label,find_anomalies(pd.DataFrame([{**base,**changes}])).anomaly_description.tolist())
print('UNKNOWN_PORTS_DETAIL',{'historical_stations_missing':s.loc[checks['unknown_station_port'],'station_id'].nunique(),'station_snapshot_stations':st.station_id.nunique()})
print('OVERLAP_SECONDS',(previous_end[overlaps]-ordered.loc[overlaps,'_start']).describe().to_dict())
print('ZERO_ENERGY_RATES_BY_YEAR',pd.DataFrame({'year':pd.to_datetime(start,unit='s',utc=True).dt.year,'zero':checks['zero_energy_session_5min']}).groupby('year').zero.mean().mul(100).round(3).to_dict())
print('ACTIVE_HIGH_POWER',pd.DataFrame({'active_avg_kw':energy/(charging/3600),'session_avg_kw':energy/(elapsed/3600),'charging_hours':charging/3600,'elapsed_hours':elapsed/3600})[checks['active_avg_power_gt7_5min']].to_dict('records'))
print('UNFLAGGED_CANDIDATES',{k:int((v & ~s.session_id.isin(calc.session_id)).sum()) for k,v in checks.items() if k in ['session_duration_diff_elapsed_60sec','zero_energy_session_5min','overlapping_port_session']})
print('FILE_HASHES',{p.name:hashlib.sha256(p.read_bytes()).hexdigest() for p in (ROOT/'data').glob('*.csv')})


PROFILE {"rows": 91518, "period_start": "2019-11-06 15:28:08+00:00", "period_end": "2026-04-02 15:20:38+00:00", "stations": 32, "ports": 40, "alarms": 23494, "current_rules": {"User actively charging for longer than 12 hours": 236, "User plugged in for longer than 24 hours": 171, "Charging power exceeds 7 kW": 1}, "unique_flagged_sessions": 405}
MATCH User actively charging for longer than 12 hours True
MATCH User plugged in for longer than 24 hours True
MATCH Charging power exceeds 7 kW True
STATION_CAPACITY [{'model': 'CT4020-HD2-GW-LTE', 'voltage': 240, 'current': 30, 'power': 6.6}, {'model': 'CT4020-HD-GW', 'voltage': 240, 'current': 30, 'power': 6.6}, {'model': 'CT4020-HD', 'voltage': 240, 'current': 30, 'power': 6.6}]
STATION_DUP_KEYS 0
CHECKS {"duplicate_session_rows": {"count": 0, "percent": 0.0}, "missing_session_id": {"count": 0, "percent": 0.0}, "invalid_start": {"count": 0, "percent": 0.0}, "invalid_end": {"count": 0, "percent": 0.0}, "invalid_energy": {"count": 0, "percent

## Findings and recommended changes

The three current rules reproduce the saved 408 anomaly rows (405 unique sessions): 171 session durations >=24 hours, 236 active charging durations >=12 hours, and one session-average power >7 kW. Saved values agree, allowing floating-point tolerance for power.

The power rule uses elapsed plugged-in time instead of active charging duration. It is a valid session-average metric, but is mislabeled as charging power. Active-time average >7 kW over at least five charging minutes identifies two sessions (10.954 and 8.711 kW); idle time hides the second under the current calculation. Neither average establishes instantaneous peak power. Station power metadata needs historical validation: 36,974 sessions (40.4%) do not match the current station/port snapshot. Existing matched ports report 6.6 kW. Do not apply a universal power ceiling or impute unmatched capacities.

Current code silently suppresses invalid durations, negative energy, reversed timestamps, and positive energy in zero-duration sessions; synthetic tests demonstrate this. These specific bad values were not observed in the snapshot. Exact threshold descriptions should say at least 24/12 hours, because the comparisons include equality.

Candidate checks: 47 duration disagreements >60 seconds (21 >5 minutes; maximum about 7.95 days); 21 sessions overlapping an earlier session on their port (9 by >60 seconds); 391 zero-energy sessions lasting >=5 minutes; 462 sessions with >=12 hours of noncharging occupancy; 5,515 same-pseudonymous-driver/same-port restarts within five minutes. Restarts may be legitimate, and overlapping sessions may reflect reporting delays; use severity, duration tolerances and corroborating alarms. Noncharging occupancy is not necessarily post-charge idle time. Zero-energy >=5-minute rates vary from 0.0% in the partial 2019 period to 0.739% in 2023; 2026 is partial. This does not establish a current incident or causal trend.

Prioritize schema/parse checks, duration consistency, tolerant port overlaps, and zero-energy sessions. Add station-specific capacity checks only where historical metadata is reliable. Report rule_id, category, severity, observed value, threshold, units, rule version, and evidence timestamps. Preserve legacy anomaly fields for compatibility. Correlate candidate failures with fault/unreachable alarms, excluding fault-cleared events from fault counts. Rolling increases in failed attempts by port are a future contextual check, not validated here.

The incremental downloader only refreshes revisited session IDs. This audit predates version 2. The pipeline now labels historical results ver=1 and replaces only revisited sessions with ver=2 results. A full historical rebuild is optional, not required; mixed-version comparisons must account for which rules ran.

## Driver messages

Inspected all 32 operations and schema element names in the current Canadian API 5.1 WSDL. No driver feedback/inbox retrieval method is exposed. customMessageText is an input to setWaitlistDone, an outbound waitlist notification. getAlarms contains technical alarm records, not driver feedback. Separate/private APIs or account-specific exports remain unverified; ask ChargePoint whether a supported Canadian driver-feedback export or endpoint exists.

Sources: https://webservices-ca.chargepoint.com/cp_api_5.1.wsdl and https://docs.chargepoint.com/ref-docs-sec/content/pdfs/4-software/api/cp_api5.1.pdf . Source scripts: lib/anomalies.py and download_chargepoint-data.py. All calculations below use data/Sessions.csv, data/Stations.csv, data/Alarms.csv and data/Anomalies.csv.
